# 02 · Etiquetas de informes con Qwen

Convierte los 4,407 informes multilingües en 12 pseudo-etiquetas y compara las 58 etiquetas gold. Con Qwen 7B tarda alrededor de 1–3 horas y necesita una GPU de 24 GB o más. Guarda cada batch, así que se puede reanudar.

In [ ]:
# CONFIG
RUN_NAME = "report_labels_v1"
RAW_ROOT = "/workspace/data/raw"
MODEL_PATH = "/workspace/models/Qwen2.5-7B-Instruct"
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
LABEL_VERSION = "v1"
BATCH_SIZE = 8
MAX_NEW_TOKENS = 350


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import torch
from tqdm.auto import tqdm

repo = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "knee").exists())
sys.path.insert(0, str(repo))
from src.knee.config import LABELS
from src.knee.labels import build_report_prompt, evaluate_pseudo_labels, parse_label_json

def read_train_csv(path):
    for encoding in ("utf-8", "cp1252", "latin-1"):
        try:
            return pd.read_csv(path, encoding=encoding)
        except UnicodeDecodeError:
            pass
    raise RuntimeError(f"No se pudo leer {path}")

train = read_train_csv(Path(RAW_ROOT) / "train.csv")
output = Path("/workspace/data/labels") / f"report_labels_{LABEL_VERSION}.csv"
output.parent.mkdir(parents=True, exist_ok=True)


In [ ]:
model_path = Path(MODEL_PATH)
if not (model_path / "config.json").exists():
    print(f"Descargando {MODEL_ID} en {model_path}. Solo hace falta una vez.")
    from huggingface_hub import snapshot_download
    snapshot_download(MODEL_ID, local_dir=model_path)
else:
    print(f"Usando modelo local: {model_path}")


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True, padding_side="left")
model = AutoModelForCausalLM.from_pretrained(
    model_path, torch_dtype=torch.bfloat16, device_map="auto", local_files_only=True
)
model.eval()


In [ ]:
done = set()
if output.exists():
    previous = pd.read_csv(output)
    done = set(previous.StudyInstanceUID.astype(str))
    print(f"Reanudando: {len(done)} informes ya procesados")
pending = train[~train.StudyInstanceUID.astype(str).isin(done)].reset_index(drop=True)

for start in tqdm(range(0, len(pending), BATCH_SIZE)):
    batch = pending.iloc[start:start + BATCH_SIZE]
    prompts = [tokenizer.apply_chat_template(
        [{"role": "user", "content": build_report_prompt(str(report))}],
        tokenize=False, add_generation_prompt=True
    ) for report in batch.Report]
    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=4096).to(model.device)
    with torch.inference_mode():
        generated = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    new_tokens = generated[:, inputs.input_ids.shape[1]:]
    answers = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
    rows = []
    for uid, answer in zip(batch.StudyInstanceUID.astype(str), answers):
        try:
            values = parse_label_json(answer)
            error = ""
        except Exception as exc:
            values = {label: 0.5 for label in LABELS}
            error = str(exc)
        rows.append({"StudyInstanceUID": uid, **values, "parse_error": error, "raw_response": answer})
    pd.DataFrame(rows).to_csv(output, mode="a", header=not output.exists(), index=False)

print(f"Listo: {output}")


In [ ]:
pseudo = pd.read_csv(output)
gold = train.dropna(subset=LABELS, how="all")
evaluation = evaluate_pseudo_labels(gold, pseudo)
display(evaluation)
print(f"Respuestas con error de JSON: {(pseudo.parse_error.fillna('') != '').sum()}")
